In [1]:
from google_play_scraper import reviews, Sort
import pandas as pd

def scrape_reviews(app_id, count=1000):
    all_reviews = []
    result, _ = reviews(
        app_id,
        lang='en',
        country='in',
        sort=Sort.NEWEST,  # Get newest first
        count=count       # Total reviews to fetch
    )
    all_reviews.extend(result)
    
    df = pd.DataFrame(all_reviews)
    return df

# Scrape Flipkart & Snapdeal
flipkart_df = scrape_reviews('com.flipkart.android', count=2000)
snapdeal_df = scrape_reviews('com.snapdeal.main', count=2000)

# Add company name column
flipkart_df['company'] = 'Flipkart'
snapdeal_df['company'] = 'Snapdeal'

# Combine & Save
final_df = pd.concat([flipkart_df, snapdeal_df], ignore_index=True)
final_df.to_csv('playstore_reviews.csv', index=False)

print("✅ Scraping complete. Saved to playstore_reviews.csv")


✅ Scraping complete. Saved to playstore_reviews.csv


In [2]:
final_df['year'] = pd.to_datetime(final_df['at']).dt.year
final_df = final_df[(final_df['year'] >= 2012) & (final_df['year'] <= 2022)]
final_df.to_csv('playstore_reviews_2012_2022.csv', index=False)
print("✅ Filtered reviews from 2012 to 2022. Saved to playstore_reviews_2012_2022.csv")

✅ Filtered reviews from 2012 to 2022. Saved to playstore_reviews_2012_2022.csv


In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import time

# ---------- CONFIG ----------
COMPANY = "Flipkart"
CATEGORIES = {
    "Electronics": "https://www.flipkart.com/electronics-store",
    "Fashion": "https://www.flipkart.com/clothing-and-accessories/pr?sid=clo",
    "Home & Furniture": "https://www.flipkart.com/furniture-store",
}

# ---------- SCRAPER ----------
def scrape_flipkart():
    # Launch Chrome (make sure Chrome + chromedriver are installed)
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")   # run without opening browser
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)
    results = []

    for category, url in CATEGORIES.items():
        driver.get(url)
        time.sleep(5)  # wait for page to load

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Extract product count (look for text like "Showing 1 – 40 of XXXX results")
        num_products = None
        count_tag = soup.find("span", string=lambda t: t and "of" in t and "results" in t.lower())
        if count_tag:
            text = count_tag.get_text(strip=True)
            try:
                num_products = int(text.split("of")[-1].split("results")[0].replace(",", "").strip())
            except:
                num_products = None

        results.append({
            "company_name": COMPANY,
            "category": category,
            "num_products": num_products,
            "date": datetime.date.today().isoformat()
        })

    driver.quit()
    return results

# ---------- RUN ----------
data = scrape_flipkart()
df = pd.DataFrame(data)
print(df)
df.to_csv('flipkart_product_counts.csv', index=False)
print("✅ Scraping complete. Saved to flipkart_product_counts.csv")

  company_name          category num_products        date
0     Flipkart       Electronics         None  2025-08-21
1     Flipkart           Fashion         None  2025-08-21
2     Flipkart  Home & Furniture         None  2025-08-21
✅ Scraping complete. Saved to flipkart_product_counts.csv


In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import time

# ---------- CONFIG ----------
COMPANY = "Flipkart"
CATEGORIES = {
    "Electronics": "https://www.flipkart.com/electronics-store",
    "Fashion": "https://www.flipkart.com/clothing-and-accessories/pr?sid=clo",
    "Home & Furniture": "https://www.flipkart.com/furniture-store",
}

def scrape_flipkart():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    results = []

    for category, url in CATEGORIES.items():
        driver.get(url)
        time.sleep(5)  # wait for page to load

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Try method 1: Look for "Showing 1 – 40 of XXXX results"
        num_products = None
        count_tag = soup.find("span", string=lambda t: t and "of" in t and "results" in t.lower())
        if count_tag:
            text = count_tag.get_text(strip=True)
            try:
                num_products = int(text.split("of")[-1].split("results")[0].replace(",", "").strip())
            except:
                num_products = None

        # Try method 2: Count product tiles if total not found
        if num_products is None:
            product_tiles = soup.find_all("div", {"class": ["_1AtVbE", "_2kHMtA", "_4ddWXP"]})
            num_products = len(product_tiles) if product_tiles else 0

        results.append({
            "company_name": COMPANY,
            "category": category,
            "num_products": num_products,
            "date": datetime.date.today().isoformat()
        })

    driver.quit()
    return results


# ---------- RUN ----------
data = scrape_flipkart()
df = pd.DataFrame(data)
print(df)

# Save to CSV
df.to_csv("flipkart_product_counts_fixed.csv", index=False)


  company_name          category  num_products        date
0     Flipkart       Electronics             0  2025-08-21
1     Flipkart           Fashion             0  2025-08-21
2     Flipkart  Home & Furniture             0  2025-08-21


In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import time

# ---------- CONFIG ----------
COMPANY = "Flipkart"
CATEGORIES = {
    "Electronics": "https://www.flipkart.com/electronics-store",
    "Fashion": "https://www.flipkart.com/clothing-and-accessories/pr?sid=clo",
    "Home & Furniture": "https://www.flipkart.com/furniture-store",
}

def scrape_flipkart():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    results = []

    for category, url in CATEGORIES.items():
        driver.get(url)
        time.sleep(5)  # wait for page load

        total_products = 0
        page_num = 1

        while True:
            time.sleep(3)
            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Find product tiles
            product_tiles = soup.find_all("div", {"class": ["_1AtVbE", "_2kHMtA", "_4ddWXP"]})
            count_this_page = len(product_tiles)
            total_products += count_this_page

            print(f"{category} - Page {page_num}: {count_this_page} products")

            # Try to click 'Next' button
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
                driver.execute_script("arguments[0].click();", next_btn)
                page_num += 1
                time.sleep(3)
            except:
                # No next button = last page
                break

        results.append({
            "company_name": COMPANY,
            "category": category,
            "num_products": total_products,
            "date": datetime.date.today().isoformat()
        })

    driver.quit()
    return results


# ---------- RUN ----------
data = scrape_flipkart()
df = pd.DataFrame(data)
print(df)

# Save to CSV
df.to_csv("flipkart_product_counts_full.csv", index=False)


Electronics - Page 1: 0 products
Fashion - Page 1: 0 products
Home & Furniture - Page 1: 0 products
  company_name          category  num_products        date
0     Flipkart       Electronics             0  2025-08-21
1     Flipkart           Fashion             0  2025-08-21
2     Flipkart  Home & Furniture             0  2025-08-21


In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import time

# ---------- CONFIG ----------
COMPANY = "Flipkart"
CATEGORIES = {
    "Electronics": "https://www.flipkart.com/electronics-store",
    "Fashion": "https://www.flipkart.com/clothing-and-accessories/pr?sid=clo",
    "Home & Furniture": "https://www.flipkart.com/furniture-store",
}

def scrape_flipkart(max_pages=3):   # scrape only first 3 pages for demo
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    results = []

    for category, url in CATEGORIES.items():
        driver.get(url)
        time.sleep(5)

        total_products = 0
        page_num = 1

        while page_num <= max_pages:
            time.sleep(3)
            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Count product cards (Flipkart uses multiple patterns)
            product_cards = (
                soup.select("div._1fQZEK") +   # large product cards (mobiles, appliances)
                soup.select("a.s1Q9rs") +     # small cards (fashion, etc.)
                soup.select("a.IRpwTa") +     # product links
                soup.select("div._4ddWXP")    # alternative small product cards
            )

            count_this_page = len(product_cards)
            total_products += count_this_page

            print(f"{category} - Page {page_num}: {count_this_page} products")

            # Try to go to next page
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")
                driver.execute_script("arguments[0].click();", next_btn)
                page_num += 1
                time.sleep(3)
            except:
                break  # no more pages

        results.append({
            "company_name": COMPANY,
            "category": category,
            "num_products": total_products,
            "date": datetime.date.today().isoformat()
        })

    driver.quit()
    return results


# ---------- RUN ----------
data = scrape_flipkart(max_pages=3)  # scrape first 3 pages to test
df = pd.DataFrame(data)
print(df)

# Save to CSV
df.to_csv("flipkart_product_counts_cards.csv", index=False)


Electronics - Page 1: 0 products
Fashion - Page 1: 0 products
Home & Furniture - Page 1: 0 products
  company_name          category  num_products        date
0     Flipkart       Electronics             0  2025-08-21
1     Flipkart           Fashion             0  2025-08-21
2     Flipkart  Home & Furniture             0  2025-08-21


In [2]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

CATEGORIES = {
    "Mobiles": "https://www.flipkart.com/mobiles",
    "Laptops": "https://www.flipkart.com/laptops",
    "Televisions": "https://www.flipkart.com/televisions",
}

async def get_count(url, page):
    await page.goto(url)
    await page.wait_for_timeout(2000)
    soup = BeautifulSoup(await page.content(), "html.parser")
    span = soup.find("span", string=lambda s: s and "of" in s and "results" in s)
    return span.get_text(strip=True) if span else "Not Found"

async def main():
    async with async_playwright() as wp:
        browser = await wp.chromium.launch(headless=True)
        page = await browser.new_page()
        for name, url in CATEGORIES.items():
            count = await get_count(url, page)
            print(f"{name}: {count}")
        await browser.close()

# In Jupyter/Colab, just use:
await main()


NotImplementedError: 

In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import csv
import datetime

# ---------- CONFIG ----------
FLIPKART_URLS = [
    "https://www.flipkart.com/samsung-galaxy-m14-5g/p/itmabc12345",  # replace with real product link
    "https://www.flipkart.com/nike-running-shoes/p/itmdef67890"
]
OUTPUT_FILE = "flipkart_pricing.csv"


# ---------- SCRAPER ----------
async def fetch_page_content(url, browser):
    page = await browser.new_page()
    await page.goto(url, timeout=60000)
    await page.wait_for_timeout(3000)  # wait for content to load
    content = await page.content()
    await page.close()
    return content


def parse_flipkart(html):
    soup = BeautifulSoup(html, "html.parser")

    # Product name
    product_name = soup.find("span", {"class": "B_NuCI"})
    product_name = product_name.get_text(strip=True) if product_name else "N/A"

    # Category breadcrumbs
    category = " > ".join([c.get_text(strip=True) for c in soup.find_all("a", {"class": "_2whKao"})]) or "N/A"

    # Price
    price = soup.find("div", {"class": "_30jeq3"})
    price = price.get_text(strip=True).replace("₹", "").replace(",", "") if price else "N/A"

    # Discount
    discount = soup.find("div", {"class": "_3Ay6Sb"})
    discount = discount.get_text(strip=True).replace("off", "").strip() if discount else "0%"

    return product_name, category, price, discount


async def main():
    today = datetime.date.today().isoformat()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["company_name", "product_name", "category", "price", "discount_percent", "date"])

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)

            for url in FLIPKART_URLS:
                try:
                    html = await fetch_page_content(url, browser)
                    product_name, category, price, discount = parse_flipkart(html)

                    writer.writerow(["Flipkart", product_name, category, price, discount, today])
                    print(f"[OK] {product_name} - ₹{price} ({discount})")

                except Exception as e:
                    print(f"[ERROR] {url}: {e}")

            await browser.close()


if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(main())



RuntimeError: asyncio.run() cannot be called from a running event loop